In [ ]:
#!/usr/bin/env python3

import asyncio
import json
import logging
import os
import smtplib
import subprocess
import sys
import time
from datetime import datetime
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from greeclimate.device import Device, DeviceInfo

#  FILE PATHS
DESKTOP_PATH    = "/home/aiteam/Desktop"
JSON_FILE       = os.path.join(DESKTOP_PATH, "ac_devices.json")
DISCOVER_SCRIPT = os.path.join(DESKTOP_PATH, "ac_discove.py")
LOG_FILE        = os.path.join(DESKTOP_PATH, "ac_automation.log")

#  CONFIGURATION only edit this block

# Only these two MACs will ever be used. All other ACs on the network
# are ignored completely, regardless of what discovery finds.
AC_PRIMARY_MAC = "502cc6000d85"   # AC1 primary unit   (started first on boot)
AC_BACKUP_MAC  = "f4911ee4b2bb"   # AC2 backup unit    (used on fault / rotation)

TEMP_THRESHOLD   = 27  # °C if temp rises above this while AC is running,
                          #       wait COOLING_WAIT_MIN then confirm fault
COOLING_WAIT_MIN = 15    # min grace period: time given to AC to cool the room
STABLE_CHECK_MIN = 10    # min how often to poll temp while AC is running fine
ROTATION_HOURS   = 5   # h   rotate ACs after this runtime (both healthy)

# Email
EMAIL_ENABLED  = True
SMTP_SERVER    = "send.one.com"
SMTP_PORT      = 587
EMAIL_FROM     = "SNMP_UPS@almuhandis.com"
EMAIL_TO       = "SNMP_UPS@almuhandis.com"
EMAIL_PASSWORD = "Admo@12qw"

#  LOGGING  (console + file)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler(LOG_FILE, encoding="utf-8"),
    ],
)
log = logging.getLogger("AC")

#  BME280 TEMPERATURE SENSOR
try:
    import bme280
    import smbus2

    _bus      = smbus2.SMBus(1)
    _cal      = bme280.load_calibration_params(_bus, 0x76)
    BME280_OK = True
    log.info("BME280 initialised on I2C bus 1, address 0x76.")

    def _raw_read():
        try:
            return bme280.sample(_bus, 0x76, _cal).temperature
        except Exception as exc:
            log.warning(f"BME280 read error: {exc}")
            return None

except ImportError:
    BME280_OK = False
    _mock     = 20.0
    log.warning("BME280 library not found MOCK temperature mode active.")

    def _raw_read():
        global _mock
        _mock = _mock + 0.4 if _mock < 30.0 else 20.0
        log.info(f"[MOCK] temperature = {_mock:.1f}°C")
        return _mock


def read_temp(fallback=25.0):
    """Return sensor temperature, or fallback if sensor errors."""
    val = _raw_read()
    if val is None:
        log.warning(f"Sensor read failed using fallback {fallback}°C")
        return fallback
    return val


#  EMAIL
def send_email(subject, body):
    """Fire-and-forget email alert. Errors are logged, never raised."""
    if not EMAIL_ENABLED:
        log.info(f"[Email disabled] {subject}")
        return
    try:
        msg            = MIMEMultipart()
        msg["From"]    = EMAIL_FROM
        msg["To"]      = EMAIL_TO
        msg["Subject"] = subject
        msg.attach(MIMEText(body, "plain"))
        with smtplib.SMTP(SMTP_SERVER, SMTP_PORT) as srv:
            srv.starttls()
            srv.login(EMAIL_FROM, EMAIL_PASSWORD)
            srv.send_message(msg)
        log.info(f"Email sent: {subject}")
    except Exception as exc:
        log.error(f"Email FAILED: {exc}")


#  DEVICE DISCOVERY / JSON
def load_devices():
    """Load full AC device list from JSON. Returns [] on any error."""
    try:
        with open(JSON_FILE) as f:
            return json.load(f)
    except Exception:
        return []


def run_discovery():
    """Run ac_discove.py subprocess and return the refreshed device list."""
    log.info("Running AC discovery to refresh IPs...")
    try:
        r = subprocess.run(
            [sys.executable, DISCOVER_SCRIPT],
            capture_output=True, text=True, timeout=25,
        )
        if r.returncode == 0:
            log.info("Discovery completed OK.")
            return load_devices()
        log.error(f"Discovery exited {r.returncode}: {r.stderr.strip()}")
    except Exception as exc:
        log.error(f"Discovery error: {exc}")
    return []


def _find_by_mac(device_list, mac):
    """Return the device dict whose MAC matches, or None."""
    for d in device_list:
        if d["mac"].lower().replace(":", "") == mac.lower().replace(":", ""):
            return d
    return None


def get_devices():
    """
    Return exactly [PRIMARY, BACKUP] filtered by MAC address.
    Order in JSON does not matter Ã¢â‚¬â€œ we always build the list as:
        index 0 = AC_PRIMARY_MAC   (AC1)
        index 1 = AC_BACKUP_MAC    (AC2)
    All other ACs on the network are ignored.
    Runs discovery automatically when either MAC is missing from JSON.
    """
    all_devs = load_devices()
    primary  = _find_by_mac(all_devs, AC_PRIMARY_MAC)
    backup   = _find_by_mac(all_devs, AC_BACKUP_MAC)

    if not primary or not backup:
        log.info(
            "One or both server room ACs not found in JSON Ã¢â‚¬â€œ "
            "running discovery..."
        )
        all_devs = run_discovery()
        primary  = _find_by_mac(all_devs, AC_PRIMARY_MAC)
        backup   = _find_by_mac(all_devs, AC_BACKUP_MAC)

    if not primary:
        log.error(f"PRIMARY AC (MAC: {AC_PRIMARY_MAC}) not found after discovery!")
    if not backup:
        log.error(f"BACKUP  AC (MAC: {AC_BACKUP_MAC}) not found after discovery!")

    if primary and backup:
        # Always index 0 = primary, index 1 = backup regardless of JSON order
        return [primary, backup]

    return []   # one or both missing caller will retry


def runtime_str(start_t):
    """Return human-readable runtime like '3h 07m'."""
    if start_t is None:
        return "N/A"
    s = int(time.time() - start_t)
    h, m = divmod(s // 60, 60)
    return f"{h}h {m:02d}m"


#  START THE BEST AVAILABLE AC  (used at boot and after fault switch)
async def start_best_ac(devices, ac_healthy, prefer=0):
    """
    Turn on the best available healthy AC.
    Tries 'prefer' first, then the other one.
    Returns the index of the AC that was successfully started, or None
    if both are unreachable (hardware problem).

    NOTE: Does NOT turn off anything first. Only called when
    no AC is currently running.
    """
    order = [prefer, 1 - prefer]
    for idx in order:
        if not ac_healthy[idx]:
            log.info(f"  Skipping AC{idx+1} (marked FAILED).")
            continue
        log.info(f"  Trying AC{idx+1}...")
        if await ac_on_verified(devices, idx):
            log.info(f"  AC{idx+1} started successfully.")
            return idx
        else:
            log.error(f"  AC{idx+1} unreachable even after discovery marking FAILED.")
            ac_healthy[idx] = False
            send_email(
                subject=f"WARNING: AC{idx+1} Unreachable at Startup",
                body=(
                    f"AC{idx+1} (MAC: {devices[idx]['mac']}) did not respond\n"
                    f"to an ON command during startup / fault recovery.\n"
                    f"Discovery was run to refresh the IP but the unit\n"
                    f"remained unreachable.\n\n"
                    f"Time: {datetime.now()}\n"
                ),
            )
    return None   # both failed


#  SWITCH TO OTHER AC  (fault recovery turn off failed first, then turn on other)
async def switch_to_other_ac(devices, ac_healthy, current_idx, reason):
    """
    Turn off the current AC first, then turn on the other AC.
    If the other AC is unreachable, try to turn the current one back on.
    Returns (new_active_idx, success_bool).
    """
    other = 1 - current_idx
    log.warning(f"Switching from AC{current_idx+1} to AC{other+1} reason: {reason}")

    if not ac_healthy[other]:
        log.error(
            f"AC{other+1} is marked FAILED cannot switch. "
            f"Keeping AC{current_idx+1} running."
        )
        return current_idx, False

    # Turn off the failed AC first
    if await ac_on_verified(devices, other):
        await asyncio.sleep(5)
        confirmed = await ac_off_verified(devices, current_idx)
        if not confirmed:
            log.warning(
                f"AC{current_idx+1} OFF not confirmed."
                f" Both ACs may be running. Please inspect."
            )
        log.info(f"Switch complete. AC{other+1} is now running.")
        return other, True
    else:
        # Other AC unreachable try to turn the original back on
        log.error(
            f"AC{other+1} did not respond during switch trying to restart AC{current_idx+1}."
        )
        ac_healthy[other] = False

        if await ac_on_verified(devices, current_idx):
            log.warning(f"Successfully restarted AC{current_idx+1}.")
            send_email(
                subject=f"WARNING: AC{other+1} Unreachable During Fault Switch",
                body=(
                    f"System tried to switch from AC{current_idx+1} to AC{other+1}.\n"
                    f"Reason for switch: {reason}\n\n"
                    f"AC{other+1} (MAC: {devices[other]['mac']}) did not respond.\n"
                    f"AC{current_idx+1} has been restarted and is running.\n\n"
                    f"Time: {datetime.now()}\n"
                ),
            )
            return current_idx, False
        else:
            # Both ACs are now off - critical situation
            log.critical(f"BOTH ACs OFF! Cannot restart either unit.")
            send_email(
                subject="CRITICAL: Both ACs Off After Failed Switch",
                body=(
                    f"CRITICAL ALERT\n\n"
                    f"After turning off AC{current_idx+1} and attempting to start\n"
                    f"AC{other+1}, both units are now off.\n\n"
                    f"Current temperature may be rising!\n\n"
                    f"Time: {datetime.now()}\n"
                ),
            )
            return None, False




async def ac_on_verified(devices, idx, max_attempts=3):
    """
    Send ON command and verify the AC actually turned on.
    Retries up to max_attempts times.
    Returns True if confirmed ON, False if all attempts failed.
    """
    label = f"AC{idx+1}"
    mac   = devices[idx]["mac"]

    for attempt in range(1, max_attempts + 1):
        log.info(f"  --> {label} ON attempt {attempt}/{max_attempts} "
                 f"(MAC:{mac[-6:]}  IP:{devices[idx]['ip']})")
        try:
            info = DeviceInfo(
                ip=devices[idx]["ip"], mac=devices[idx]["mac"],
                name=f"AC-{devices[idx]['mac'][-6:]}", port=7000,
            )
            ac = Device(info)
            await ac.bind()

            # Send ON command
            ac.power = True
            await ac.push_state_update()

            # Wait 2 seconds then read back the status to verify
            await asyncio.sleep(2)
            await ac.update_state()

            if ac.power:
                log.info(f"  {label} ON confirmed (attempt {attempt})")
                return True
            else:
                log.warning(f"  {label} still OFF after attempt {attempt} retrying...")

        except Exception as exc:
            # IP may have changed try discovery on first failure
            if attempt == 1:
                log.warning(f"  {label} unreachable running discovery to refresh IP...")
                fresh_all = run_discovery()
                refreshed = _find_by_mac(fresh_all, mac)
                if refreshed:
                    devices[idx] = refreshed
                    log.info(f"  {label} new IP = {devices[idx]['ip']} retrying...")
            else:
                log.warning(f"  {label} ON attempt {attempt} failed: {exc}")

        await asyncio.sleep(3)  # wait before next attempt

    log.error(f"  {label} could NOT be confirmed ON after {max_attempts} attempts")
    send_email(
        subject=f"WARNING: AC{idx+1} Did Not Confirm ON Command",
        body=(
            f"AC{idx+1} (MAC: {devices[idx]['mac']}) was sent an ON command\n"
            f"{max_attempts} times but could not confirm it turned on.\n\n"
            f"PLEASE CHECK PHYSICALLY that AC{idx+1} is on.\n\n"
            f"Time: {datetime.now()}\n"
        ),
    )
    return False


async def ac_off_verified(devices, idx, max_attempts=3):
    """
    Send OFF command and verify the AC actually turned off.
    Retries up to max_attempts times.
    Returns True if confirmed OFF, False if all attempts failed.
    """
    label = f"AC{idx+1}"

    for attempt in range(1, max_attempts + 1):
        log.info(f"  --> {label} OFF attempt {attempt}/{max_attempts} "
                 f"(MAC:{devices[idx]['mac'][-6:]}  IP:{devices[idx]['ip']})")
        try:
            info = DeviceInfo(
                ip=devices[idx]["ip"], mac=devices[idx]["mac"],
                name=f"AC-{devices[idx]['mac'][-6:]}", port=7000,
            )
            ac = Device(info)
            await ac.bind()

            # Send OFF command
            ac.power = False
            await ac.push_state_update()

            # Wait 2 seconds then read back the status to verify
            await asyncio.sleep(2)
            await ac.update_state()

            if not ac.power:
                log.info(f"  {label} OFF confirmed (attempt {attempt})")
                return True
            else:
                log.warning(f"  {label} still ON after attempt {attempt} retrying...")

        except Exception as exc:
            log.warning(f"  {label} OFF attempt {attempt} failed: {exc}")

        await asyncio.sleep(3)  # wait before next attempt

    log.error(f"  {label} could NOT be confirmed OFF after {max_attempts} attempts")
    send_email(
        subject=f"WARNING: AC{idx+1} Did Not Confirm OFF Command",
        body=(
            f"AC{idx+1} (MAC: {devices[idx]['mac']}) was sent an OFF command\n"
            f"{max_attempts} times but could not confirm it turned off.\n\n"
            f"PLEASE CHECK PHYSICALLY that AC{idx+1} is off.\n\n"
            f"Time: {datetime.now()}\n"
        ),
    )
    return False

#  MAIN
async def main():
    """
    DESIGN PRINCIPLE: active_ac is NEVER None after the boot phase.
    At least 1 AC must be running at all times.

    Boot sequence:
      1. Load/discover devices, filter to server room MACs only
      2. Start PRIMARY AC (index 0) immediately regardless of temperature
      3. Enter the monitoring loop

    Monitoring loop Ã¢â‚¬â€œ each iteration:
      A. Refresh device list silently (picks up DHCP IP changes by MAC)
      B. Read temperature
      C. If temp > threshold: wait grace period Ã¢â€ â€™ confirm fault Ã¢â€ â€™ switch
      D. If temp OK: check rotation timer Ã¢â€ â€™ sleep
    """

    ac_healthy = [True, True]   # [AC1_primary_ok, AC2_backup_ok]
    last_health_reset = time.time()  # Track when we last reset health flags

    log.info("=" * 68)
    log.info("  SERVER ROOM AC AUTOMATION  v3.1  Ã¢â‚¬â€œ  STARTED")
    log.info(f"  Primary AC MAC        : {AC_PRIMARY_MAC}")
    log.info(f"  Backup  AC MAC        : {AC_BACKUP_MAC}")
    log.info(f"  Temp fault threshold  : {TEMP_THRESHOLD} Ã‚Â°C")
    log.info(f"  Fault grace period    : {COOLING_WAIT_MIN} min")
    log.info(f"  Stable poll interval  : {STABLE_CHECK_MIN} min")
    log.info(f"  Rotation interval     : {ROTATION_HOURS:.0f} h (only when both ACs healthy)")
    log.info(f"  Health reset interval : 12 h (to retry previously failed ACs)")
    log.info(f"  Sensor                : {'BME280 real hardware' if BME280_OK else 'MOCK test mode'}")
    log.info("=" * 68)

    # BOOT PHASE find server room ACs by MAC, then start primary
    log.info("[BOOT] Loading device list...")
    devices = []

    while len(devices) < 2:
        devices = get_devices()
        if len(devices) < 2:
            log.error(
                f"Could not find both server room ACs "
                f"(PRIMARY:{AC_PRIMARY_MAC}  BACKUP:{AC_BACKUP_MAC}). "
                f"Retrying in 30 s..."
            )
            await asyncio.sleep(30)

    log.info(f"[BOOT] AC1 (PRIMARY): MAC={devices[0]['mac']}  IP={devices[0]['ip']}")
    log.info(f"[BOOT] AC2 (BACKUP) : MAC={devices[1]['mac']}  IP={devices[1]['ip']}")

    # Start primary AC immediately. Server room must always have cooling.
    log.info("[BOOT] Starting PRIMARY AC (AC1) immediately (always-on policy)...")
    active_ac = await start_best_ac(devices, ac_healthy, prefer=0)

    if active_ac is None:
        # Both ACs unreachable at boot Ã¢â‚¬â€œ critical situation
        log.critical("[BOOT] CRITICAL: Could not start ANY AC at boot!")
        send_email(
            subject="CRITICAL: AC Automation Ã¢â‚¬â€œ Cannot Start Any AC at Boot",
            body=(
                f"The AC automation system started but was UNABLE to\n"
                f"turn on any AC unit.\n\n"
                f"AC1 PRIMARY (MAC: {devices[0]['mac']}) : unreachable\n"
                f"AC2 BACKUP  (MAC: {devices[1]['mac']}) : unreachable\n\n"
                f"Both health flags have been reset. The system will\n"
                f"keep retrying every 60 seconds.\n\n"
                f"Time: {datetime.now()}\n"
            ),
        )
        while active_ac is None:
            log.info("[BOOT RETRY] Trying to start an AC...")
            await asyncio.sleep(60)
            devices = get_devices()
            ac_healthy[:] = [True, True]
            active_ac = await start_best_ac(devices, ac_healthy, prefer=0)

    ac_start_t = time.time()
    log.info(f"[BOOT] AC{active_ac+1} is running. Entering monitoring loop.")

    #  MONITORING LOOP
    while True:

        # Refresh device list silently each iteration
        # get_devices() always returns [PRIMARY, BACKUP] by MAC,
        # so even if DHCP changed an IP the correct unit is still at
        # index 0 and index 1.
        fresh = get_devices()
        if len(fresh) == 2:
            devices = fresh

        #Read temperature
        temp    = read_temp()
        runtime = (time.time() - ac_start_t) / 3600.0
        time_since_health_reset = (time.time() - last_health_reset) / 3600.0

        log.info(
            f"[{time.strftime('%H:%M:%S')}]  "
            f"Temp: {temp:.1f}Ã‚Â°C  |  "
            f"Active: AC{active_ac+1}  Runtime: {runtime_str(ac_start_t)}  |  "
            f"Health: AC1={'OK  ' if ac_healthy[0] else 'FAIL'}  "
            f"AC2={'OK  ' if ac_healthy[1] else 'FAIL'}"
        )

        # HEALTH RESET every 12 hours, reset failed flags to retry ACs
        if time_since_health_reset >= 12.0:
            log.info("=" * 60)
            log.info("12-HOUR HEALTH RESET: Clearing failed AC flags to retry units")
            log.info(f"Previous health state: AC1={'OK' if ac_healthy[0] else 'FAIL'}, AC2={'OK' if ac_healthy[1] else 'FAIL'}")

            # Reset both health flags to True
            ac_healthy_old = ac_healthy.copy()
            ac_healthy[:] = [True, True]
            last_health_reset = time.time()

            log.info(f"New health state: AC1=OK, AC2=OK")

            # Send notification about health reset
            if not ac_healthy_old[0] or not ac_healthy_old[1]:
                send_email(
                    subject="INFO: AC Health Flags Reset for Retry",
                    body=(
                        f"The system has reset the AC health flags after 12 hours.\n\n"
                        f"Previous health state:\n"
                        f"  AC1: {'OK' if ac_healthy_old[0] else 'FAILED'}\n"
                        f"  AC2: {'OK' if ac_healthy_old[1] else 'FAILED'}\n\n"
                        f"Both ACs will now be considered healthy and available for use.\n\n"
                        f"Time: {datetime.now()}\n"
                    ),
                )
            log.info("=" * 60)

        #  FAULT CHECK Temperature is HIGH while AC is running
        if temp > TEMP_THRESHOLD:
            log.warning(
                f"TEMP {temp:.1f}Ã‚Â°C > {TEMP_THRESHOLD}Ã‚Â°C while AC{active_ac+1} running! "
                f"Starting {COOLING_WAIT_MIN}-min grace period..."
            )

            # Wait, give the AC a chance to catch up (load spike / startup lag)
            await asyncio.sleep(COOLING_WAIT_MIN * 60)
            temp = read_temp()
            log.info(
                f"Temp after {COOLING_WAIT_MIN}-min grace: {temp:.1f}Ã‚Â°C  "
                f"(threshold {TEMP_THRESHOLD}Ã‚Â°C)"
            )

            if temp <= TEMP_THRESHOLD:
                log.info(
                    f"Temp recovered to {temp:.1f}Ã‚Â°C Ã¢â‚¬â€œ "
                    f"AC{active_ac+1} is fine. Was a transient spike."
                )
                await asyncio.sleep(STABLE_CHECK_MIN * 60)
                continue

            # CONFIRMED FAULT: AC running but cannot cool the room
            failed = active_ac
            log.error(
                f"FAULT CONFIRMED: AC{failed+1} ran for {runtime_str(ac_start_t)} "
                f"but temp is still {temp:.1f}Ã‚Â°C. Switching to AC{2-failed}..."
            )

            # Mark current AC failed BEFORE switching
            ac_healthy[failed] = False

            # Build honest email subject
            other_healthy = ac_healthy[1 - failed]
            email_subject = (
                f"WARNING: AC{failed+1} Cooling Failure Ã¢â‚¬â€œ "
                + (f"Switching to AC{2-failed}" if other_healthy else "NO BACKUP AVAILABLE")
            )
            send_email(
                subject=email_subject,
                body=(
                    f"AC FAULT ALERT\n\n"
                    f"AC{failed+1} (MAC: {devices[failed]['mac']}) ran for "
                    f"{runtime_str(ac_start_t)}\n"
                    f"but FAILED to keep the server room below {TEMP_THRESHOLD}Ã‚Â°C.\n\n"
                    f"Temperature now    : {temp:.1f}Ã‚Â°C\n"
                    f"AC{2-failed} healthy : {'YES Ã¢â‚¬â€œ switching now' if other_healthy else 'NO Ã¢â‚¬â€œ also failed!'}\n\n"
                    f"Please inspect AC{failed+1} immediately.\n\n"
                    f"Time: {datetime.now()}\n"
                ),
            )

            if not other_healthy:
                # Other AC already marked failed Ã¢â‚¬â€œ no switch possible
                log.critical(
                    f"BOTH ACs FAILED. No backup available. "
                    f"Resetting health and retrying in 5 min..."
                )
                send_email(
                    subject="CRITICAL: Both ACs Failed Ã¢â‚¬â€œ Server Room Overheating",
                    body=(
                        f"CRITICAL ALERT\n\n"
                        f"Both AC units are now marked as FAILED.\n\n"
                        f"AC1 PRIMARY (MAC: {devices[0]['mac']}) : FAILED\n"
                        f"AC2 BACKUP  (MAC: {devices[1]['mac']}) : FAILED\n\n"
                        f"Current temperature : {temp:.1f}Ã‚Â°C\n"
                        f"Threshold           : {TEMP_THRESHOLD}Ã‚Â°C\n\n"
                        f"IMMEDIATE PHYSICAL INSPECTION IS REQUIRED.\n"
                        f"System will reset health flags and retry in 5 minutes.\n\n"
                        f"Time: {datetime.now()}\n"
                    ),
                )
                log.info("Waiting 5 min then resetting health flags and retrying...")
                await asyncio.sleep(300)
                ac_healthy[:] = [True, True]
                await ac_off_verified(devices, failed)
                active_ac = await start_best_ac(devices, ac_healthy, prefer=0)
                if active_ac is None:
                    log.critical("Still cannot start any AC. Will retry in 60 s...")
                    await asyncio.sleep(60)
                    continue
                ac_start_t = time.time()
                continue

            # Switch to the other AC (turn off failed first, then turn on other)
            new_ac, switched = await switch_to_other_ac(
                devices, ac_healthy, failed,
                reason=f"AC{failed+1} failed to cool room (temp={temp:.1f}Ã‚Â°C)"
            )

            if switched:
                active_ac  = new_ac
                ac_start_t = time.time()
                log.info(
                    f"AC{active_ac+1} is now active. "
                    f"Rotation DISABLED (AC{failed+1} is FAILED)."
                )
            elif new_ac is None:
                # Both ACs are off - critical situation
                log.critical("Both ACs are OFF! Attempting to restart any AC...")
                active_ac = await start_best_ac(devices, ac_healthy, prefer=0)
                if active_ac is None:
                    log.critical("Cannot start any AC. Waiting 60s and retrying...")
                    await asyncio.sleep(60)
                    continue
                ac_start_t = time.time()
                continue
            else:
                # switch_to_other_ac kept us on the same AC (other was unreachable)
                log.critical(
                    f"Both ACs FAILED after fault switch attempt. "
                    f"Resetting health and retrying in 5 min..."
                )
                send_email(
                    subject="CRITICAL: Both ACs Failed During Fault Switch",
                    body=(
                        f"AC{failed+1} had a cooling fault, then AC{2-failed} "
                        f"also failed to respond during the switch.\n\n"
                        f"Current temperature : {temp:.1f}Ã‚Â°C\n"
                        f"System will reset and retry in 5 minutes.\n\n"
                        f"Time: {datetime.now()}\n"
                    ),
                )
                await asyncio.sleep(300)
                ac_healthy[:] = [True, True]
                await ac_off_verified(devices, active_ac)
                active_ac = await start_best_ac(devices, ac_healthy, prefer=0)
                if active_ac is None:
                    await asyncio.sleep(60)
                    continue
                ac_start_t = time.time()

            continue   # re-check from top immediately after any switch

        #  TEMP IS NORMAL AC is running fine
        both_healthy = ac_healthy[0] and ac_healthy[1]

        if both_healthy and runtime >= ROTATION_HOURS:
            # Time to rotate for wear-balancing
            other = 1 - active_ac
            log.info(
                f"ROTATION: AC{active_ac+1} has run {runtime:.1f}h "
                f"Ã¢â€ â€™ switching to AC{other+1} for wear balancing"
            )

            new_ac, switched = await switch_to_other_ac(
                devices, ac_healthy, active_ac,
                reason=f"scheduled {ROTATION_HOURS:.0f}h wear-balance rotation"
            )

            if switched:
                active_ac  = new_ac
                ac_start_t = time.time()
                log.info(f"Rotation complete Ã¢â‚¬â€œ AC{active_ac+1} now running.")
            else:
                log.warning(
                    f"Rotation failed Ã¢â‚¬â€œ AC{active_ac+1} continues running. "
                    f"Rotation now permanently disabled."
                )

        else:
           # sleep and re-poll
            if both_healthy:
                reason = f"{ROTATION_HOURS:.0f}h not reached yet ({runtime:.1f}h)"
            else:
                reason = "rotation disabled (one AC marked FAILED)"

            log.info(
                f"Temp OK ({temp:.1f}Ã‚Â°C), AC{active_ac+1} running fine Ã¢â‚¬â€œ {reason}. "
                f"Sleeping {STABLE_CHECK_MIN} min..."
            )
            await asyncio.sleep(STABLE_CHECK_MIN * 60)

    # end while True

if __name__ == "__main__":
    try:
        asyncio.run(main())
    except KeyboardInterrupt:
        log.info("System stopped by user (Ctrl-C). Goodbye!")